In [1]:
import pandas as pd
import glob, os

path = "Raw-data/"
tables = {}
for f in glob.glob(path + "*.csv"):
    name = os.path.basename(f).replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    tables[name] = pd.read_csv(f)

orders = tables["orders"]
reviews = tables["order_reviews"]
products = tables["products"]
translation = tables["product_category_name_translation"]

In [2]:
order_dates = ["order_purchase_timestamp", "order_approved_at",
               "order_delivered_carrier_date", "order_delivered_customer_date",
               "order_estimated_delivery_date"]
for c in order_dates:
    orders[c] = pd.to_datetime(orders[c])

for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])

print(orders.dtypes)
print(orders[order_dates].min())
print(orders[order_dates].max())

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
order_purchase_timestamp        2016-09-04 21:15:19
order_approved_at               2016-09-15 12:16:38
order_delivered_carrier_date    2016-10-08 10:34:01
order_delivered_customer_date   2016-10-11 13:46:32
order_estimated_delivery_date   2016-09-30 00:00:00
dtype: datetime64[ns]
order_purchase_timestamp        2018-10-17 17:30:18
order_approved_at               2018-09-03 17:40:06
order_delivered_carrier_date    2018-09-11 19:48:28
order_delivered_customer_date   2018-10-17 13:22:46
order_estimated_delivery_date   2018-11-12 00:00:00
dtype: datetime64[ns]


In [3]:
orders_delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

print(len(orders_delivered))

96470


In [4]:
od = orders_delivered

od["delivery_days"] = (od["order_delivered_customer_date"] - od["order_purchase_timestamp"]).dt.days

od["delay_days"] = (od["order_delivered_customer_date"].dt.normalize()
                    - od["order_estimated_delivery_date"].dt.normalize()).dt.days

od["is_late"] = od["delay_days"] > 0

print(od[["delivery_days", "delay_days"]].describe())
print("Negative delivery days:", (od["delivery_days"] < 0).sum())
print("Late %:", round(od["is_late"].mean() * 100, 1))

       delivery_days    delay_days
count   96470.000000  96470.000000
mean       12.093604    -11.875889
std         9.551380     10.182105
min         0.000000   -147.000000
25%         6.000000    -17.000000
50%        10.000000    -12.000000
75%        15.000000     -7.000000
max       209.000000    188.000000
Negative delivery days: 0
Late %: 6.8


In [5]:
reviews_clean = (
    reviews
    .sort_values(["order_id", "review_answer_timestamp"])
    .drop_duplicates(subset="order_id", keep="last")
)

print(len(reviews_clean), reviews_clean["order_id"].is_unique)

98673 True


In [6]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

products = products.merge(translation, on="product_category_name", how="left")

no_translation = products[products["product_category_name_english"].isna()]["product_category_name"].unique()
print(no_translation)

products["product_category_name_english"] = (
    products["product_category_name_english"].fillna(products["product_category_name"])
)

print(len(products), products["product_category_name_english"].isna().sum())

['unknown' 'pc_gamer' 'portateis_cozinha_e_preparadores_de_alimentos']
32951 0


In [7]:
os.makedirs("data/clean", exist_ok=True)

orders_delivered.to_csv("data/clean/orders_delivered.csv", index=False)
reviews_clean.to_csv("data/clean/reviews_clean.csv", index=False)
products.to_csv("data/clean/products_clean.csv", index=False)

for name in ["customers", "order_items", "order_payments", "sellers"]:
    tables[name].to_csv(f"data/clean/{name}.csv", index=False)

In [8]:
late_orders = orders_delivered[orders_delivered['delay_days'] > 60]
len(late_orders)

79

In [9]:
late_orders["order_purchase_timestamp"].dt.to_period("M").value_counts()

order_purchase_timestamp
2017-03    12
2017-11    11
2018-02     9
2018-01     8
2017-04     7
2017-05     5
2017-06     5
2017-12     4
2017-02     4
2018-04     3
2018-05     2
2018-03     2
2017-09     2
2017-08     2
2018-06     1
2017-10     1
2017-01     1
Freq: M, Name: count, dtype: int64

In [10]:
reviews_clean=reviews_clean.drop(columns=["review_comment_message", "review_comment_title"])

In [12]:
print(reviews_clean.columns)

Index(['review_id', 'order_id', 'review_score', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')


In [13]:
os.makedirs("data/clean", exist_ok=True)

orders_delivered.to_csv("data/clean/orders_delivered.csv", index=False)
reviews_clean.to_csv("data/clean/reviews_clean.csv", index=False)
products.to_csv("data/clean/products_clean.csv", index=False)

for name in ["customers", "order_items", "order_payments", "sellers"]:
    tables[name].to_csv(f"data/clean/{name}.csv", index=False)

print(os.listdir("data/clean"))

['customers.csv', 'orders_delivered.csv', 'order_items.csv', 'order_payments.csv', 'products_clean.csv', 'reviews_clean.csv', 'sellers.csv']
